In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from nnsight import LanguageModel
import torch as t

device = t.device('cuda:0')
dtype = t.float32
model_name = "EleutherAI/pythia-70m-deduped"

model = LanguageModel(
    model_name,
    device_map=device,
    dispatch=True,
    torch_dtype=dtype,
)

/fs01/home/xiaowenz/494/.env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print(model)

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
          (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise

In [24]:
from prompts.util import gen_response
import nnsight
gen_config = {
    'max_new_tokens': 5,
    'pad_token_id': model.tokenizer.eos_token_id,
}
# Old approach:
prompt = 'The Eiffel Tower is in the city of'
layers = model.gpt_neox.layers
n_new_tokens = 3
hidden_states = []

with model.generate(prompt, max_new_tokens=n_new_tokens) as tracer:
    total = 0
    hidden_states = nnsight.list().save() # Initialize & .save() nnsight list
    layers.all()

        # Apply intervention - set first layer output to zero
    layers[0].output[0][:] = 0

    # Append desired hidden state post-intervention
    hidden_states.append(layers[-1].output[0]) # no need to call .save
    # Don't need to loop or call .next()!

    out = model.generator.output.save()
print(len(hidden_states))
for h in hidden_states:
    print(h.shape)

model.tokenizer.decode(out[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


3
torch.Size([1, 10, 512])
torch.Size([1, 1, 512])
torch.Size([1, 1, 512])


'The Eiffel Tower is in the city of\n\n\n'

In [23]:
print(layers)

ModuleList(
  (0-5): 6 x GPTNeoXLayer(
    (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (post_attention_dropout): Dropout(p=0.0, inplace=False)
    (post_mlp_dropout): Dropout(p=0.0, inplace=False)
    (attention): GPTNeoXAttention(
      (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
      (dense): Linear(in_features=512, out_features=512, bias=True)
    )
    (mlp): GPTNeoXMLP(
      (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
      (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
      (act): GELUActivation()
    )
  )
)
